In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hashlib
import json
import time
from collections.abc import Callable
from dataclasses import dataclass, field

# Permissioned chains — a chain where the guest list matters

This notebook is a companion to the Medium article [Introducing Permission Chains](https://medium.com/mitb-for-all/introducing-permissioned-chains-1d8a71f5b09a)

Builds on notebooks 1–3. Quick recap of the story so far, because this
notebook is about to change nearly everything you've gotten used to:

| Notebook | Question it answers | Mechanism |
| --- | --- | --- |
| 1. Mining | How do we make history tamper-evident, and rewriting *expensive*? | Hash-chained blocks + proof-of-work |
| 2. Proof of Stake | Who gets to propose the next block, without burning a small country's electricity bill? | Stake-weighted lottery + slashing |
| 3. Merkle trees | How do we commit to a *huge* batch of data cheaply, and prove one item belongs without exposing the rest? | Binary hash tree + O(log N) proofs |
| **4. This notebook** | What if we *don't* want just anyone to join, there's no token to mine or stake, and several orgs that don't fully trust each other still need to agree on one shared ledger? | Membership + endorsement + ordering + commit-time validation |

Notebooks 1–3 were all, deep down, the same shape: **one open network,
one rule for "who may write next"** (find a nonce / win the stake
lottery), and **one validator loop** that re-checks the *whole chain*
from genesis whenever asked. Fun fact: nobody ever had to prove who they
*were* — a hash doesn't care about your identity, only your CPU cycles
or your stake.

That's exactly wrong for, say, three banks running a shared settlement
ledger. They don't want:

- randoms joining the network,
- a proof-of-work energy bill or a speculative staking token,
- or a system where "51% of the CPU" or "51% of the stake" decides the
  truth, when the actual participants are a known, small, named list of
  institutions who'd very much like a court to be able to point at who
  signed what.

This notebook builds a **permissioned (consortium) chain**, modelled
loosely on **Hyperledger Fabric**. The core shift: instead of "whoever
wins gets to propose, then everyone re-verifies," Fabric-style chains
run **execute → endorse → order → validate** — multiple named
organizations *independently execute and agree* on a transaction's
outcome *before* it ever gets sequenced into a block.

**What we build here (toy model):**

1. A `MembershipService` — gatekeeping who's even allowed to hold a
   peer identity (no entry, no participation, full stop)
2. `transfer_chaincode` — the actual "smart contract" business logic
3. `Peer.endorse()` — named orgs *execute* a proposed transaction and
   sign off on the result, before anyone tries to order it
4. `OrderingService` — a dumb, opinion-free sequencer (it does **not**
   validate anything)
5. `PermissionedBlockchain.add_ordered_block()` /
   `validate_and_commit_block()` — cutting a block is separate from
   judging what's in it: every ordered transaction lands in a block,
   and only afterward do commit-time checks (endorsement policy
   satisfied? endorsers agree? and **MVCC**, multi-version concurrency
   control — is the world state still what the endorsers thought it
   was?) decide whether each one is applied to state
6. `propose_with_retry()` — the client-side retry loop for when two
   transactions collide over the same piece of state

> **Honest scope:** this is a single-process simulation of Fabric's
> *shape*, not Fabric itself. No real PKI/X.509 identities, no chaincode
> containers, no channels or private data collections, no Raft/BFT
> consensus inside the ordering service, and no network partitions. What
> *is* faithful: the execute-before-order pipeline, and the reason MVCC
> conflict detection exists at all. Treat this the way you'd treat a
> floor plan — accurate about the rooms and doors, not architecturally
> load-bearing.

The toy model is ours. The mechanisms are not — see **Sources** at the end.


## Gate #1: Membership — permission to even show up

In notebooks 1–3, "who can participate" was answered by physics or
economics: anyone with a CPU can mine (PoW), anyone can lock up tokens
and enter the lottery (PoS). There was no admissions office.

Here there is one. `MembershipService` is a deliberately tiny stand-in
for what Fabric calls an **MSP (Membership Service Provider)** — in real
life backed by X.509 certificates issued by a CA that each consortium
member trusts. No entry in the registry, no participation:
`Peer.endorse()` checks `is_registered()` before doing anything else,
and throws if you're not on the list.

This is the permissioned chain's first and most fundamental departure
from notebooks 1–3: **identity is a precondition, not an emergent
property of effort or stake.**

In [3]:
def sha256(data: str) -> str:
    """Return the hex-encoded SHA-256 digest of a UTF-8 encoded string."""
    return hashlib.sha256(data.encode()).hexdigest()


class MembershipService:
    """Toy stand-in for Fabric's MSP: tracks which orgs are registered participants."""

    def __init__(self) -> None:
        """Initialize an empty registry of member organizations."""
        self.registered_orgs: set[str] = set()

    def register(self, org_name: str) -> None:
        """Add an organization to the registry, granting it participant identity."""
        print(f"  [MembershipService] Registered org: {org_name}")
        self.registered_orgs.add(org_name)

    def is_registered(self, org_name: str) -> bool:
        """Return True if the given organization is a registered participant."""
        return org_name in self.registered_orgs

## The ledger's state: a versioned key-value world, not just a list of blocks

Notebooks 1–3 derived "current state" by replaying the chain (or just
mutated a `Validator.stake` field directly). That's fine for a toy with
one writer at a time.

Here, multiple orgs can *concurrently* propose transactions against the
same shared state, so we need to answer a sharper question than "what's
the current balance?" — we need to ask **"what version of this key was
I looking at when I computed my answer?"**

`WorldState` tracks two things per key:

| Field | Role |
| --- | --- |
| `balances` | The actual current values (a normal key-value store) |
| `versions` | A counter, bumped every time a key is written |

Hang onto `version_of()` — it's the hook the whole optimistic-concurrency
story further down is built on. `ExecutionResult` is what a chaincode
run produces: a `write_set` (what it wants to change) and
`read_versions` (what versions it *assumed* were current when it
computed that write set). Neither is applied to `WorldState` yet — this
is a **proposal**, not a commit.

In [4]:
@dataclass
class WorldState:
    """Versioned key-value ledger state: current balances plus a per-key write version counter."""

    balances: dict[str, float] = field(default_factory=dict)
    versions: dict[str, int] = field(default_factory=dict)

    def version_of(self, key: str) -> int:
        """Return the current version number of a key, or 0 if it has never been written."""
        return self.versions.get(key, 0)


@dataclass
class ExecutionResult:
    """The proposed effect of a chaincode run: what it wants to write, and the versions it read."""

    write_set: dict[str, float]
    read_versions: dict[str, int]

## Chaincode: the business logic, run speculatively

"Chaincode" is Fabric's word for what other platforms call a **smart
contract**. `transfer_chaincode` is about as simple as one gets: check
a balance, compute a new pair of balances.

The important part isn't the arithmetic — it's *when* this runs.
Executing this function does **not** touch the real ledger. It runs
against a state snapshot, tentatively, and returns an `ExecutionResult`
that some `Peer` will sign off on. Two different peers, running this
same function against the same state, had better get the *same*
`write_set` — that agreement is what "endorsement" is actually checking
in a moment.

In [5]:
def transfer_chaincode(state: WorldState, frm: str, to: str, amount: float) -> ExecutionResult:
    """Chaincode: speculatively compute the balances resulting from a transfer, without mutating state.

    Args:
        state: The world state snapshot to read balances/versions from.
        frm: Account to debit.
        to: Account to credit.
        amount: Amount to transfer.

    Returns:
        An ExecutionResult with the proposed write_set and the versions read.

    Raises:
        ValueError: If `frm` has insufficient funds.
    """
    if state.balances.get(frm, 0) < amount:
        raise ValueError(f"Insufficient funds: {frm} has {state.balances.get(frm, 0)}, needs {amount}")
    return ExecutionResult(
        write_set={frm: state.balances.get(frm, 0) - amount, to: state.balances.get(to, 0) + amount},
        read_versions={frm: state.version_of(frm), to: state.version_of(to)},
    )

## Execute → Endorse → Order → Validate

This is the shape swap. Compare it to notebooks 1–3's loop of
*(one node proposes) → (everyone re-checks a completed block)*:

| Phase | Who does it | What happens |
| --- | --- | --- |
| **Execute** | Each endorsing peer, independently | Run the chaincode against *their own* view of state; get a candidate `write_set` |
| **Endorse** | Same peers | Sign off — here, just record a hash of the `write_set` — *without* touching the shared ledger |
| **Order** | `OrderingService` | Sequence proposals into a canonical order. No business logic, no opinion on correctness — just a queue |
| **Validate** | `PermissionedBlockchain.validate_and_commit_block()` | *Now* check, per transaction in the block: enough endorsers? do they agree? is the state they read still current? An invalid transaction stays in the block, marked so — it just never touches `WorldState` |

Why bother separating execute from order, when notebooks 1–3 just did
"propose, mine/sign, done"? Because here **multiple mutually
distrusting organizations** are involved. You can't let one party
compute the result and have everyone else take their word for it — so
you make several of them compute it *independently* and check they
land on the same answer, and you do that check **before** wasting
everyone's time sequencing it.

`Endorsement` intentionally carries only a **hash** of the write set,
not the write set itself — the full result travels alongside in
`ExecutionResult` for this toy, but in real Fabric this separation is
what lets the ordering service sequence transactions **without ever
seeing their contents** (useful when channels carry sensitive data).

`OrderingService.order()` is almost insultingly simple on purpose: it
gets paid to put things in a queue, and has zero opinion on whether
what's in the queue is honest. In real Fabric this is itself a cluster
running Raft (or historically, Kafka) — consensus on *ordering*, kept
completely separate from consensus on *correctness*.

Notice that **Order** and **Validate** are two separate method calls on
`PermissionedBlockchain` below — `add_ordered_block()` and
`validate_and_commit_block()` — not one. A block gets cut the moment a
batch of proposals is ordered, full stop; whether anything inside it is
actually legitimate is a question `validate_and_commit_block()` answers
afterward, transaction by transaction.

In [6]:
@dataclass
class Endorsement:
    """A peer's sign-off on a chaincode execution, carrying only a hash of the write set."""

    peer_name: str
    result_hash: str


@dataclass
class Peer:
    """A named, registered organization's node, capable of independently executing and endorsing chaincode."""

    name: str
    membership: MembershipService

    def endorse(self, chaincode: Callable, state: WorldState, *args) -> tuple[Endorsement, ExecutionResult]:
        """Execute chaincode against the given state and sign off on the resulting write set.

        Args:
            chaincode: The chaincode function to run.
            state: The world state to execute against.
            *args: Positional arguments forwarded to `chaincode`.

        Returns:
            A tuple of (Endorsement, ExecutionResult) for the proposed transaction.

        Raises:
            PermissionError: If this peer's org is not a registered participant.
        """
        if not self.membership.is_registered(self.name):
            raise PermissionError(f"{self.name} is not a registered participant.")
        result = chaincode(state, *args)
        result_hash = sha256(json.dumps(result.write_set, sort_keys=True))
        return Endorsement(peer_name=self.name, result_hash=result_hash), result


@dataclass
class ProposedTransaction:
    """A candidate transaction awaiting ordering and commit-time validation."""

    description: str
    endorsements: list[Endorsement]
    execution_result: ExecutionResult


class OrderingService:
    """A deliberately dumb sequencer: puts proposals in order, without judging their correctness."""

    def __init__(self, name: str) -> None:
        """Initialize the ordering service with a name used in log output."""
        self.name = name

    def order(self, proposals: list[ProposedTransaction]) -> list[ProposedTransaction]:
        """Sequence a batch of proposed transactions into a canonical order."""
        print(f"  [{self.name}] Sequencing {len(proposals)} proposal(s).")
        return list(proposals)

## Finally, the ledger itself

One more shape change before the `Block` and `PermissionedBlockchain`
classes below: there's no `nonce` (no mining, notebook 1) and no
`proposer`/`stake` (no lottery, notebook 2). Instead there's an
`orderer` field — a name, not a competitor in any race, just a record
of which ordering service sequenced this block.

There's also, this time, a real batch of `transactions` inside each
block — not a single opaque summary string — plus a parallel
`validation_codes` list, one slot per transaction. That split matters:
cutting a block happens the moment transactions are ordered; deciding
whether each transaction in it is legitimate happens afterward, in
`PermissionedBlockchain.validate_and_commit_block()`, not in the
block's constructor.

In [7]:
class Block:
    """A tamper-evident block carrying a batch of ordered transactions, each pending its own validation code."""

    def __init__(
        self,
        index: int,
        transactions: list[ProposedTransaction],
        previous_hash: str,
        orderer: str,
    ) -> None:
        """Create and hash a new block from an already-ordered batch of transactions.

        Args:
            index: The block's position in the chain.
            transactions: The ordered batch of proposed transactions this block carries.
            previous_hash: The hash of the preceding block.
            orderer: Name of the ordering service that sequenced this block.
        """
        self.index = index
        self.timestamp = time.time()
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.orderer = orderer

        # Validation happens after ordering.
        self.validation_codes = ["NOT_VALIDATED"] * len(transactions)

        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        """Compute this block's SHA-256 hash over its index, timestamp, transactions, previous hash, and orderer."""
        tx_contents = [
            {
                "description": tx.description,
                "write_set": tx.execution_result.write_set,
                "read_versions": tx.execution_result.read_versions,
                "endorsements": [
                    {
                        "peer_name": e.peer_name,
                        "result_hash": e.result_hash,
                    }
                    for e in tx.endorsements
                ],
            }
            for tx in self.transactions
        ]

        block_contents = json.dumps(
            {
                "index": self.index,
                "timestamp": self.timestamp,
                "transactions": tx_contents,
                "previous_hash": self.previous_hash,
                "orderer": self.orderer,
            },
            sort_keys=True,
        )
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def __repr__(self) -> str:
        """Return a human-readable summary of the block, including per-transaction validation codes."""
        return (
            f"Block #{self.index}: "
            f"{len(self.transactions)} transaction(s), "
            f"validation={self.validation_codes}\n"
        )

### A noticeably boring `Block` — until you look at what's inside

Notice what's *missing* compared to earlier notebooks: no proof-of-work
target, no proposer eligibility field. `hash` and `previous_hash` are
still here — tamper-evidence within a block and links between blocks
haven't gone anywhere. What's new is `transactions`, a real batch (not
a single opaque summary string), and `validation_codes`, one slot per
transaction, all initialized to `"NOT_VALIDATED"` at construction time.

That's the tell: unlike notebooks 1 and 2, a block here gets created
*before* anyone has judged whether what's in it is valid.
`compute_hash()` only ever looks at `transactions`, never
`validation_codes` — so a block's identity is fixed the moment it's
ordered, and validating it afterward can update `validation_codes`
without changing the block's own hash. The main event,
`PermissionedBlockchain.validate_and_commit_block()` below, runs once
per block, checking every transaction inside it individually — not by
re-scanning the whole chain after the fact the way `is_valid()` did in
notebooks 1 and 2.

In [8]:
class PermissionedBlockchain:
    """A consortium ledger where blocks are cut by ordering, then validated (and applied to state) after the fact."""

    def __init__(
        self,
        required_endorsers: set[str],
        min_endorsements: int,
        orderer_name: str,
    ) -> None:
        """Initialize the chain with an empty genesis block and an empty world state.

        Args:
            required_endorsers: Org names that must all endorse a transaction for it to be valid.
            min_endorsements: Minimum number of endorsements required, regardless of policy.
            orderer_name: Name of the ordering service recorded on each block.
        """
        self.required_endorsers = required_endorsers
        self.min_endorsements = min_endorsements
        self.orderer_name = orderer_name

        self.chain: list[Block] = [
            Block(
                index=0,
                transactions=[],
                previous_hash="0" * 64,
                orderer=orderer_name,
            )
        ]

        self.state = WorldState()

    def add_ordered_block(
        self,
        transactions: list[ProposedTransaction],
    ) -> Block:
        """Cut a new block from an already-ordered batch of transactions and append it to the chain.

        This is the ordering-service side of the pipeline: it has no opinion
        on whether any transaction in the batch is actually valid.

        Args:
            transactions: The ordered batch of proposed transactions to include.

        Returns:
            The newly created, appended block.
        """
        previous_block = self.chain[-1]

        block = Block(
            index=len(self.chain),
            transactions=transactions,
            previous_hash=previous_block.hash,
            orderer=self.orderer_name,
        )

        self.chain.append(block)

        print(
            f"  [{self.orderer_name}] "
            f"Created Block #{block.index} with "
            f"{len(transactions)} transaction(s)."
        )

        return block

    def validate_and_commit_block(self, block: Block) -> list[bool]:
        """Validate every transaction in a block and apply valid ones to world state.

        For each transaction, checks (in order): the endorsement policy is
        satisfied, a minimum number of endorsements was collected, all
        endorsers agree on the result, and no read key's version has changed
        since the transaction was endorsed (MVCC). The block's
        `validation_codes` are updated in place; the block itself is never
        removed or re-sequenced, even if every transaction in it turns out
        invalid -- only the *state-changing effect* of invalid transactions
        is withheld.

        Args:
            block: An already-ordered, chain-appended block to validate.

        Returns:
            A list of booleans, one per transaction, in the same order as
            `block.transactions`.
        """
        results: list[bool] = []

        for i, tx in enumerate(block.transactions):
            print(f"  Validating: {tx.description}")

            endorser_names = {e.peer_name for e in tx.endorsements}

            if not endorser_names.issuperset(self.required_endorsers):
                print("    INVALID: missing required endorsers")
                block.validation_codes[i] = "INVALID_ENDORSEMENT_POLICY"
                results.append(False)
                continue

            if len(tx.endorsements) < self.min_endorsements:
                print("    INVALID: not enough endorsements")
                block.validation_codes[i] = "INVALID_QUORUM"
                results.append(False)
                continue

            if len({e.result_hash for e in tx.endorsements}) > 1:
                print("    INVALID: endorsers disagree")
                block.validation_codes[i] = "INVALID_ENDORSER_MISMATCH"
                results.append(False)
                continue

            mvcc_conflict = False

            for key, read_version in tx.execution_result.read_versions.items():
                current_version = self.state.version_of(key)

                if current_version != read_version:
                    print(
                        f"    INVALID: MVCC CONFLICT on '{key}' -- "
                        f"endorsed against version {read_version}, "
                        f"current is {current_version}"
                    )
                    block.validation_codes[i] = "INVALID_MVCC_CONFLICT"
                    results.append(False)
                    mvcc_conflict = True
                    break

            if mvcc_conflict:
                continue

            for key, new_value in tx.execution_result.write_set.items():
                self.state.balances[key] = new_value
                self.state.versions[key] = self.state.version_of(key) + 1

            block.validation_codes[i] = "VALID"
            results.append(True)

            print("    VALID: write set committed to WorldState")

        return results

## Order first, validate second: `add_ordered_block()` then `validate_and_commit_block()`

This is where "permissioned" earns its keep, and where this notebook
now visibly splits into two what a simpler design might fuse into one
step:

- **`add_ordered_block()`** does the *ordering service's* job: take
  whatever batch of transactions arrived, cut a `Block` around them,
  append it to the chain. It has zero opinion on whether any of those
  transactions are legitimate.
- **`validate_and_commit_block()`** does the *committer's* job: walk
  every transaction in that already-appended block and, for each one,
  check:

  1. **Endorsement policy** — did *all* `required_endorsers` actually
     sign off? (Here: all three banks, always — a strict "everyone
     must agree" policy. Real Fabric policies can be richer, e.g. "any
     2 of 3".)
  2. **Quorum** — at least `min_endorsements` signatures, full stop.
  3. **Agreement** — do all endorsers' `result_hash`es match? If two
     honest-looking endorsers computed *different* write sets,
     something is wrong (bug, malicious peer, or stale state) — mark
     it invalid rather than guess who's right.
  4. **MVCC conflict check** — for every key the transaction *read*, is
     `state.version_of(key)` still what the endorsers assumed? If
     someone else's transaction already landed and bumped that
     version, this proposal is stale — mark it invalid, no matter how
     honest it was.

  Only a transaction that passes all four gets its write set applied
  to `WorldState`, with its `validation_codes` entry set to `"VALID"`;
  everything else gets a specific `INVALID_*` code — and stays on the
  chain anyway, permanently, as a record that it was proposed and
  rejected. **The block itself is never rolled back or re-cut just
  because something inside it turned out invalid** — that's the whole
  point of separating "cut the block" from "judge its contents," and
  it's exactly how a real Fabric committer behaves: a block full of
  nothing but invalid transactions is still a valid block.

> **Honest scope gap, named on purpose:** unlike notebooks 1 and 2, this
> `PermissionedBlockchain` has no `is_valid()` that re-walks the whole
> chain checking every `hash`/`previous_hash` link from genesis. Each
> block *could* still be re-verified that way — the fields are all
> there — we just don't wire it up here, because the interesting new
> idea in this notebook is commit-time transaction validation, not
> chain replay. Bolting that check back on would be a straightforward
> exercise, not a gap in the idea.

### Then: what happens when two transactions collide?

Gate #4 above is where **optimistic concurrency control (OCC)** lives.
"Optimistic" because nobody locks anything up front — every peer just
executes hopefully, assuming nothing else will change the state before
their transaction lands. Most of the time that's true, and it's free
efficiency. When it's *not* true, `validate_and_commit_block()` catches
the stale read at commit time, marks that transaction
`INVALID_MVCC_CONFLICT`, and leaves the block on the chain anyway — and
`propose_with_retry()` below is the client-side response: shrug, re-run
`execute` against whatever the state actually is *now*, get ordered
into a fresh block, and try again.

In [9]:
def propose_with_retry(
    description: str,
    peers: list[Peer],
    chaincode: Callable,
    chain: PermissionedBlockchain,
    orderer: OrderingService,
    args: tuple,
    max_attempts: int = 3,
) -> bool:
    """The client-side retry loop -- what a real Fabric SDK does
    automatically. On an MVCC conflict, don't give up: re-run EXECUTE
    against whatever the CURRENT state now is, and try again. This is
    the 'optimistic' half of optimistic concurrency control -- pay the
    retry cost only on an actual collision, not on every transaction.

    Every attempt still gets ordered into its own block and validated,
    even a failing one -- a rejected attempt leaves an INVALID block
    behind on the chain, it just never touches WorldState.

    Args:
        description: Label for this transaction.
        peers: Endorsing peers to use.
        chaincode: The smart contract function to run.
        chain: The ledger to submit to.
        orderer: The ordering service to sequence through.
        args: Arguments to pass to the chaincode.
        max_attempts: How many times to retry before giving up.

    Returns:
        True if eventually committed, False if all attempts were exhausted.
    """
    for attempt in range(1, max_attempts + 1):
        print(f"  Attempt {attempt}: {description} (fresh EXECUTE against current state)")
        endorsements: list[Endorsement] = []
        execution_result: ExecutionResult | None = None
        for peer in peers:
            endorsement, result = peer.endorse(chaincode, chain.state, *args)
            endorsements.append(endorsement)
            execution_result = result
        tx = ProposedTransaction(description, endorsements, execution_result)
        ordered = orderer.order([tx])
        block = chain.add_ordered_block(ordered)
        results = chain.validate_and_commit_block(block)
        if results[0]:
            return True
        print("    -> stale, retrying...\n")
    print(f"  Gave up after {max_attempts} attempts.")
    return False


## Demo: three banks, one shared ledger, zero shared trust

The scenario: `Bank-A`, `Bank-B`, and `Bank-C` jointly run a settlement
ledger. None of them trusts the other two's arithmetic, so the
endorsement policy requires **all three** to independently compute and
agree on every transaction before it commits. Nobody is mining. Nobody
is staking. Everybody is, presumably, billing hours.

In [10]:
print("=== Setup ===\n")
membership = MembershipService()
membership.register("Bank-A")
membership.register("Bank-B")
membership.register("Bank-C")
peer_a = Peer("Bank-A", membership)
peer_b = Peer("Bank-B", membership)
peer_c = Peer("Bank-C", membership)
peers = [peer_a, peer_b, peer_c]

chain = PermissionedBlockchain(
    required_endorsers={"Bank-A", "Bank-B", "Bank-C"},
    min_endorsements=3,
    orderer_name="Shared-Ordering-Service",
)
orderer = OrderingService("Shared-Ordering-Service")
chain.state.balances = {"Alice-Account": 1000.0, "Bob-Account": 500.0, "Carol-Account": 0.0}
print(f"  Starting balances: {chain.state.balances}\n")

=== Setup ===

  [MembershipService] Registered org: Bank-A
  [MembershipService] Registered org: Bank-B
  [MembershipService] Registered org: Bank-C
  Starting balances: {'Alice-Account': 1000.0, 'Bob-Account': 500.0, 'Carol-Account': 0.0}



### Here comes the race condition

Two clients submit **concurrently**: Alice pays Bob $200, and — before
that lands anywhere — Alice pays Carol $150. Both get endorsed against
the *same* starting version of `Alice-Account` (version 0), because
neither peer knew about the other proposal yet. Only one of these can
possibly be valid once both try to land, since together they'd overdraw
Alice. Watch `add_ordered_block()` cut a block for *each* of them
regardless, and `validate_and_commit_block()` sort out which one
actually gets applied to state — by version number, not by which one
"seems more honest."

In [11]:
print("=== Two clients endorse CONCURRENTLY, both against Alice-Account version 0 ===\n")
e_a1, res1 = peer_a.endorse(transfer_chaincode, chain.state, "Alice-Account", "Bob-Account", 200.0)
e_b1, _ = peer_b.endorse(transfer_chaincode, chain.state, "Alice-Account", "Bob-Account", 200.0)
e_c1, _ = peer_c.endorse(transfer_chaincode, chain.state, "Alice-Account", "Bob-Account", 200.0)
tx1 = ProposedTransaction("Alice pays Bob $200", [e_a1, e_b1, e_c1], res1)

e_a2, res2 = peer_a.endorse(transfer_chaincode, chain.state, "Alice-Account", "Carol-Account", 150.0)
e_b2, _ = peer_b.endorse(transfer_chaincode, chain.state, "Alice-Account", "Carol-Account", 150.0)
e_c2, _ = peer_c.endorse(transfer_chaincode, chain.state, "Alice-Account", "Carol-Account", 150.0)
tx2_stale_attempt = ProposedTransaction("Alice pays Carol $150", [e_a2, e_b2, e_c2], res2)
print("  All three endorsed. Neither peer knew about the other proposal.\n")

=== Two clients endorse CONCURRENTLY, both against Alice-Account version 0 ===

  All three endorsed. Neither peer knew about the other proposal.



In [12]:
print("=== ORDER + VALIDATE: Tx1 happens to get sequenced first ===\n")
ordered1 = orderer.order([tx1])
block1 = chain.add_ordered_block(ordered1)
chain.validate_and_commit_block(block1)

=== ORDER + VALIDATE: Tx1 happens to get sequenced first ===

  [Shared-Ordering-Service] Sequencing 1 proposal(s).
  [Shared-Ordering-Service] Created Block #1 with 1 transaction(s).
  Validating: Alice pays Bob $200
    VALID: write set committed to WorldState


[True]

In [13]:
print("\n=== Tx2's ORIGINAL (now-stale) endorsement finally reaches ORDER + VALIDATE ===\n")
ordered2 = orderer.order([tx2_stale_attempt])
block2 = chain.add_ordered_block(ordered2)
results = chain.validate_and_commit_block(block2)
print(f"  First attempt succeeded? {results[0]}\n")


=== Tx2's ORIGINAL (now-stale) endorsement finally reaches ORDER + VALIDATE ===

  [Shared-Ordering-Service] Sequencing 1 proposal(s).
  [Shared-Ordering-Service] Created Block #2 with 1 transaction(s).
  Validating: Alice pays Carol $150
    INVALID: MVCC CONFLICT on 'Alice-Account' -- endorsed against version 0, current is 1
  First attempt succeeded? False



In [14]:
print("=== Client retries automatically, re-executing against CURRENT state ===\n")
success = propose_with_retry(
    "Alice pays Carol $150", peers, transfer_chaincode, chain, orderer,
    ("Alice-Account", "Carol-Account", 150.0),
)
print(f"\n  Eventually succeeded? {success}\n")

=== Client retries automatically, re-executing against CURRENT state ===

  Attempt 1: Alice pays Carol $150 (fresh EXECUTE against current state)
  [Shared-Ordering-Service] Sequencing 1 proposal(s).
  [Shared-Ordering-Service] Created Block #3 with 1 transaction(s).
  Validating: Alice pays Carol $150
    VALID: write set committed to WorldState

  Eventually succeeded? True



In [15]:
print("=== Final state ===")
print(f"  Balances: {chain.state.balances}")
print(f"  Versions: {chain.state.versions}")
print(f"  Chain length: {len(chain.chain)} blocks")

=== Final state ===
  Balances: {'Alice-Account': 650.0, 'Bob-Account': 700.0, 'Carol-Account': 150.0}
  Versions: {'Alice-Account': 2, 'Bob-Account': 1, 'Carol-Account': 1}
  Chain length: 4 blocks


---

## Takeaways

1. **Membership is a precondition here, not an emergent property** —
   compare to notebooks 1–2, where "who can participate" fell out of
   physics (CPU) or economics (stake).
2. **Execute happens before order** — several named orgs independently
   run the chaincode and must *agree*, before anything gets sequenced.
   Notebooks 1–3 only ever had one party compute a result at a time.
3. **Cutting a block and validating it are two different moments** —
   `add_ordered_block()` appends whatever was ordered, no questions
   asked; `validate_and_commit_block()` judges it afterward, one
   transaction at a time. An invalid transaction doesn't vanish or
   rewind the chain — it stays put, marked `INVALID_*`, just never
   applied to `WorldState`.
4. **The ordering service is deliberately dumb** — sequencing and
   correctness-checking are different jobs, done by different
   components, so a single dishonest or broken orderer can't forge
   agreement it never actually confirmed.
5. **MVCC replaces "51% of something" as the anti-cheating mechanism** —
   there's no hash puzzle and no stake at risk; conflicting transactions
   are caught by comparing state *versions* at commit time.
6. **Optimistic concurrency control** trades a small chance of wasted
   work (a rejected, retried transaction, still permanently visible on
   the chain) for never having to lock shared state up front — cheap
   when collisions are rare.

### The whole series, side by side

| | 1. PoW | 2. PoS | 3. Merkle trees | 4. Permissioned |
| --- | --- | --- | --- | --- |
| Who can join | Anyone with a CPU | Anyone who stakes | N/A (a data structure, not a network) | Only registered members |
| Right to write | Win the hash race | Win the stake lottery | — | Endorsement policy satisfied |
| What stops cheating | Re-mining is expensive | Slashing destroys stake | Collision-resistant hashing | MVCC + multi-party agreement |
| Validation happens | Re-scan whole chain | Re-scan whole chain | Recompute path to root | Per-transaction, after the block is already cut |
| Real-world cousin | Bitcoin | Ethereum (post-Merge) | Certificate Transparency, tx batches | Hyperledger Fabric, R3 Corda |

**Still out of scope (on purpose), if you want to keep digging:** real
PKI-backed membership (X.509, CAs, revocation), chaincode running in
actual isolated containers, **channels** and **private data
collections** for confidentiality between subsets of members, real
consensus *inside* the ordering service (Raft, BFT), and what happens
when an endorsing peer is not just slow but actively lying (Byzantine
behaviour — this notebook's "endorsers disagree" check catches honest
disagreement, not a coordinated lie).

Four notebooks, four different answers to the same underlying question:
**how do a bunch of people who don't fully trust each other agree on
one shared history?** Burn electricity, put money at risk, hash your way
to a cheap membership check, or — as it turns out most real enterprises
actually do — just make everyone show ID at the door.

## Sources

Membership, endorsement, ordering, and MVCC on a guest-list chain are Hyperledger Fabric's design, not a thought experiment:

- Androulaki, E., Barger, A., Bortnikov, V., et al. (2018). [Hyperledger Fabric: A Distributed Operating System for Permissioned Blockchains](https://arxiv.org/abs/1801.10228). EuroSys. Execute → endorse → order → validate, and why a permissioned OS does not need a native coin.
- Hyperledger Fabric documentation. [Transaction flow](https://hyperledger-fabric.readthedocs.io/en/latest/txflow.html). The same four steps in the words the project uses today.
- Hyperledger Fabric documentation. [Read-write sets and MVCC](https://hyperledger-fabric.readthedocs.io/en/latest/readwrite.html). Why two endorsements against the same key version cannot both commit.
